# DIP super-resolution — results summary (Set5 ×4)

Reads finished runs from Weights & Biases and builds every table used in the discussion. **It trains nothing and needs no GPU** (CPU runtime is fine, ~3–5 min), so it can run next to a training notebook.

| Section | Question it answers |
|---|---|
| A | Which fidelity loss is best? (Stage 1, corrected metric) |
| B | Do the paper's regularizers help in our setup? (75-run matrix) |
| C | Does performance depend on the downsampling operator? |
| D | Why does bicubic input fail? (fit to input vs. upsampling) |
| E | Does the regularizer gain grow with operator mismatch? (mismatch study; filled in as runs finish) |

All PSNR values are RGB PSNR of the final reconstruction against the ground-truth HR image (dB, higher is better). One seed unless stated. Training setup "adapted" = lr 0.01, input noise 0.03, 2000 iterations; the paper used lr 0.001, noise 0.01, 5500 iterations.

## Setup

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path
REPO_URL, BRANCH = "https://github.com/lakshjain7/deep-image-prior.git", "claude/epic-ride-rnqbk4"
REPO_DIR = Path("/content/deep-image-prior")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "origin", BRANCH], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
try: importlib.import_module("wandb")
except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)

import wandb, numpy as np, pandas as pd
from IPython.display import display
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass
wandb.login()

import stage3
from stage3 import IMAGES, LOSSES, REGULARIZERS
PROJECT = "dip-sr-regularizers"
W = dict(stage3.DEFAULT_REG_WEIGHTS)
tracker = stage3.Tracker(None, PROJECT, Path("/content/unused"))
api = wandb.Api()
print("W&B entity:", tracker.entity)

def latest_complete(key):
    ok = [r for r in tracker._query(key) if r.summary.get("status") == "complete"]
    return max(ok, key=lambda r: r.created_at) if ok else None

def summary_rows(jobs):
    rows = []
    for j in jobs:
        r = latest_complete(j["key"])
        if r is not None:
            rows.append(dict(protocol=j["protocol"], input=j["lr_method"], kernel=j.get("kernel", "lanczos2"), image=j["image"], loss=j["loss"],
                             reg=j["reg"], hr=r.summary["psnr_hr"], fit=r.summary.get("psnr_LR"),
                             upsample=r.summary.get("psnr_bicubic_baseline")))
    missing = [j["key"] for j in jobs if latest_complete(j["key"]) is None]
    return pd.DataFrame(rows), missing

## A. Fidelity losses (Stage 1, corrected)

Stage 1 originally averaged `psnr_lr` (how well the output re-downsamples to the LR input). Here the same 30 runs are re-scored with `psnr_HR` (ground truth). 5 images × 3 losses × seeds 7 and 123.

In [ ]:
rows = []
for run in api.runs(f"{tracker.entity}/deep-image-prior"):
    c = run.config
    if (c.get("loss") in ("l1", "l1-em", "m-l1") and c.get("num_iter") == 2000 and c.get("kernel_type") == "lanczos2"
            and c.get("factor") == 4 and c.get("lr") == 0.01 and run.state == "finished" and "psnr_HR" in run.summary):
        rows.append(dict(image=Path(str(c["image"])).stem, loss=c["loss"],
                         psnr_input_domain=run.summary["psnr_LR"], psnr_ground_truth=run.summary["psnr_HR"]))
s1 = pd.DataFrame(rows)
print("runs per image/loss (expect 2):"); display(s1.groupby(["loss", "image"]).size().unstack())
print("Mean PSNR (dB): originally reported column vs. correct column")
display(s1.groupby("loss")[["psnr_input_domain", "psnr_ground_truth"]].mean().round(2))
pi = s1.pivot_table(index="image", columns="loss", values="psnr_ground_truth")
pi["L1-EM minus L1"] = pi["l1-em"] - pi["l1"]; pi["M-L1 minus L1"] = pi["m-l1"] - pi["l1"]
print("Per image (ground-truth PSNR, mean of 2 seeds)"); display(pi.round(2))
print("Paper (Table 4, Set5): L1-EM +0.41, M-L1 +0.23 dB over L1")

## B. Regularizers in our setup (75-run matrix, PIL-Lanczos LR input)

Gains are measured the same way as the paper's Table 4: mean over the five images of PSNR minus the **L1, no-regularizer** run of the same image. Weights from the paper's Table 3 (Set5).

In [ ]:
jobs_b = stage3.build_jobs(IMAGES, LOSSES, REGULARIZERS, [7], "adapted", W)
db, miss_b = summary_rows(jobs_b)
print(f"complete {len(db)}/{len(jobs_b)}", "| missing:", miss_b)
base = db[(db.loss == "l1") & (db.reg == "none")].set_index("image")["hr"]
db["gain_vs_L1"] = db.hr - db.image.map(base)
ours = db.pivot_table(index="loss", columns="reg", values="gain_vs_L1").reindex(columns=REGULARIZERS)
paper = pd.DataFrame({"none": [0, 0.4084, 0.2287], "tv": [1.4063, 1.5608, 1.5693], "nn": [1.1539, 1.3105, 1.2803],
                      "tvssah": [1.4729, 1.6365, 1.6615], "bd": [0.7939, 1.0262, 1.0587]}, index=["l1", "l1-em", "m-l1"])
print("Ours: mean gain over L1/no-regularizer (dB)"); display(ours.round(2))
print("Paper Table 4 (Set5), same definition"); display(paper[REGULARIZERS].round(2))
own = db.merge(db[db.reg == "none"][["image", "loss", "hr"]], on=["image", "loss"], suffixes=("", "_none"))
own["gain_vs_same_loss"] = own.hr - own.hr_none
print("Ours: gain of each regularizer over the same loss without regularizer (dB)")
display(own[own.reg != "none"].pivot_table(index="loss", columns="reg", values="gain_vs_same_loss").round(2))

## C. Does the downsampling operator matter? (L1, no regularizer)

Two separate choices: how the LR image is **generated** (PIL Lanczos, or bicubic as in the authors' code) and which kernel the **model** uses (Lanczos-2 or Lanczos-3).

In [ ]:
jobs_c = []
for m in ["pil-lanczos", "bicubic"]:
    for k in ["lanczos2", "lanczos3"]:
        jobs_c += stage3.build_jobs(IMAGES, ["l1"], ["none"], [7], "adapted", W, m, k)
dc, miss_c = summary_rows(jobs_c)
tc = dc.pivot_table(index=["input", "kernel"], columns="image", values="hr")
print("PSNR vs ground truth (dB)"); display(tc.round(2))
print("Effect of the LR generation (PIL Lanczos minus bicubic, Lanczos-2 model)")
display((tc.loc[("pil-lanczos", "lanczos2")] - tc.loc[("bicubic", "lanczos2")]).round(2).to_frame("dB").T)
print("Effect of the model kernel (Lanczos-3 minus Lanczos-2)")
display(pd.DataFrame({m: tc.loc[(m, "lanczos3")] - tc.loc[(m, "lanczos2")] for m in ["pil-lanczos", "bicubic"]}).T.round(2))
print("missing:", miss_c)

## D. Why bicubic input fails

`fit` = PSNR of the downsampled output against the LR input (how well the network reproduces what it sees). `upsample` = plain bicubic upsampling of the same LR input. Healthy DIP should beat `upsample`.

In [ ]:
jobs_d = jobs_c + stage3.build_jobs(["butterfly", "head"], ["l1"], ["none", "tv"], [7], "paper", W, "bicubic")
dd, _ = summary_rows(jobs_d)
dd["DIP_minus_upsample"] = dd.hr - dd.upsample
display(dd[["protocol", "input", "kernel", "image", "reg", "fit", "hr", "upsample", "DIP_minus_upsample"]].round(2))
print("Mean DIP minus upsampling by LR input (no regularizer):")
display(dd[dd.reg == "none"].groupby(["protocol", "input"])["DIP_minus_upsample"].mean().round(2))

## E. Mismatch study: does the regularizer gain grow with operator mismatch?

L1 loss. Three LR inputs ordered from closest to the model's operator to furthest: PIL Lanczos → antialiased bicubic → bicubic without antialiasing (authors' code). TV on all five images; NN, TVSSAH and BD under bicubic on four images. Rows appear as the runs finish.

In [ ]:
order = ["pil-lanczos", "bicubic-aa", "bicubic"]
jobs_e = []
for m in order:
    jobs_e += stage3.build_jobs(IMAGES, ["l1"], ["none", "tv"], [7], "adapted", W, m)
jobs_e += stage3.build_jobs(["bird", "butterfly", "head", "woman"], ["l1"], ["nn", "tvssah", "bd"], [7], "adapted", W, "bicubic")
de, miss_e = summary_rows(jobs_e)
t = de.pivot_table(index=["input", "image"], columns="reg", values="hr")
gain = t.drop(columns="none").sub(t["none"], axis=0)
up = de[de.reg == "none"].set_index(["input", "image"])["upsample"]
print("Gain of each regularizer over no regularizer (dB)"); display(gain.round(2))
print("Mean gain by LR input"); display(gain.groupby("input").mean().reindex(order).round(2))
print("DIP minus plain upsampling (no regularizer)"); display((t["none"] - up).groupby("input").mean().reindex(order).round(2))
print(f"complete {len(de)}/{len(jobs_e)} | missing:", miss_e)